<div style="background:#081420;color:#e6edf3;padding:24px 28px;border-radius:12px;font-family:ui-sans-serif,system-ui,-apple-system,'Segoe UI',Inter,Helvetica,Arial,sans-serif">
  <div style="font-size:12px;font-weight:700;letter-spacing:0.16em;text-transform:uppercase;color:#00d4ff;margin-bottom:6px">Aintropy Solutions</div>
  <div style="font-size:24px;font-weight:600;line-height:1.2">Action-state consistency, end to end</div>
  <div style="font-size:13.5px;color:#7c8ea1;margin-top:10px;line-height:1.5">A per-clip verdict on whether pressing a key actually did what the training data claims it did. Real optical flow, real pipeline, no download needed.</div>
</div>

## Setup

One install cell. If you already have `kuration-sdk` in this environment (Binder image, dev checkout, etc.), skip. On Colab, run it once.

In [ ]:
# Install the SDK with the gameplay extras (registers pixel_shift + action_state_consistency).
# The gameplay extras include the optical-flow backend used by pixel_shift.
%pip install --quiet 'kuration-sdk[gameplay]'

## The idea in one paragraph

An action-conditioned world model learns whatever rule the training footage shows it. Press W, camera moves forward. Except sometimes it does not, because a wall was in the way, because the engine buffered the input, because the scope changed the movement math. The model has no way to tell which examples encode the rule and which encode the exception.

The `action_state_consistency` operator gives every clip a verdict. `consistent`, `inconsistent`, or `inconclusive`. It works by isolating single, uninterrupted key presses, projecting optical flow onto the axis that press is supposed to move the camera on, and comparing against a baseline built from the dataset's own typical behavior.

## A minimal synthetic dataset

This is the same pattern the SDK uses in `examples/07_action_state_consistency.py`. Frames come from an in-memory `FakeVideoProvider` that yields real texture (shifting noise). Optical flow can actually track it. Actions follow the real 51-dim DIAMOND encoding.

One 8-frame clip. One isolated W press at frame 3. Everything else at rest.

In [ ]:
import numpy as np
import kuration.gameplay  # noqa: F401 -- registers the analyzer on import
from kuration import FrameRef, SchemaSpec, VideoMetadata, input
from kuration.gameplay.pixel_shift import diamond_action_decode as dad

ACTION_DIM = dad.ACTION_WIDTH
ACTION_COLUMNS = [f"a{i}" for i in range(ACTION_DIM)]


class FakeVideoProvider:
    """Yields lazy frame refs over an in-memory frame stack."""
    capabilities = {"streaming": True, "variable_frame_rate": True}

    def __init__(self, frames):
        self.frames = frames

    def probe(self, uri):
        h, w = self.frames[0].shape[:2]
        return VideoMetadata(uri, w, h, len(self.frames), float(len(self.frames) - 1), 1.0)

    def iter_refs(self, uri, *, source_id=None):
        for i in range(len(self.frames)):
            yield FrameRef(uri, source_id or uri, i, float(i))

    def decode_batch(self, refs):
        return np.stack([self.frames[ref.frame_index] for ref in refs])


def shifting_noise_frames(n, h=40, w=60, seed=3):
    """Textured frames with a real 1px/frame shift. Optical flow needs actual texture."""
    rng = np.random.default_rng(seed)
    base = rng.integers(0, 255, size=(h, w + n, 3), dtype=np.uint8)
    return [base[:, i : i + w] for i in range(n)]


def action_vector(w_pressed=False):
    """One 51-dim DIAMOND action vector. Mouse bins must be strict one-hot at rest."""
    vec = np.zeros(ACTION_DIM, dtype=np.float64)
    if w_pressed:
        vec[dad.KEY_NAMES.index("W")] = 1.0
    mx_idx = int(np.argmin(np.abs(dad.MOUSE_X_POSSIBLES - 0.0)))
    my_idx = int(np.argmin(np.abs(dad.MOUSE_Y_POSSIBLES - 0.0)))
    vec[dad.MX_OFF + mx_idx] = 1.0
    vec[dad.MY_OFF + my_idx] = 1.0
    return vec


N_FRAMES = 8
PRESS_AT = 3
actions = [action_vector() for _ in range(N_FRAMES)]
actions[PRESS_AT] = action_vector(w_pressed=True)

rows = [
    {"t": t, **{col: float(v) for col, v in zip(ACTION_COLUMNS, actions[t], strict=True)}}
    for t in range(N_FRAMES)
]
provider = FakeVideoProvider(shifting_noise_frames(N_FRAMES))

print(f"clip has {N_FRAMES} frames, isolated W press at frame {PRESS_AT}")

## The pipeline

Three lines. Input the rows against a schema. Join a video source. Slice into windows. Then a single `action_state_consistency` call with the baselines and controls to check.

A `baseline` says: for this key press, expect pixel shift in this direction and roughly this magnitude. In real use, you compute baselines from the dataset itself (10th to 90th percentile per control cell). Here we hand-write a plausible one for W.

In [ ]:
SCHEMA = SchemaSpec.from_dict({
    "name": "action-state-example",
    "fields": {"action": {"kind": "vector", "dims": ACTION_COLUMNS}},
})

# Expected: W press produces positive shift, any magnitude for this demo.
baselines = {"W": {"1": {"expected_sign": 1, "min": 0.0, "max": 1e9}}}

pipeline = (
    input({"clip1": rows}, schema=SCHEMA, fields={"action": ACTION_COLUMNS}, time="t")
    .join_video({"clip1": "fake.mp4"}, provider=provider, on="frame_index")
    .window(4, stride=2)
)

checked = pipeline.action_state_consistency(baselines=baselines, controls=["W"])
result = checked.metadata["gameplay"]["action_state_consistency"]["result"]

print("passed:", result["passed"])
print("summary:", result["summary"])

## Flipping the baseline

Now say the baseline expects W to shift pixels in the *negative* direction (backward-looking convention, or a mislabeled control). The same footage now fails, because the measured motion is the opposite of what the baseline says it should be. Same clip, different verdict.

In [ ]:
flipped_baselines = {"W": {"1": {"expected_sign": -1, "min": 0.0, "max": 1e9}}}

flipped_checked = pipeline.action_state_consistency(baselines=flipped_baselines, controls=["W"])
flipped_result = flipped_checked.metadata["gameplay"]["action_state_consistency"]["result"]

print("passed:", flipped_result["passed"])
print("summary:", flipped_result["summary"])

## Compose downstream

The verdict lives on the pipeline. The pipeline is still just a pipeline. So a `.filter()` on the flagged windows, or a `.sample()` that weights against them, chains directly on top.

In [ ]:
# .describe() prints the exact operator chain.
print(checked.describe())

## Extending to real footage

The SDK ships a full worked example against the public TeaPearce Counter-Strike Deathmatch dataset at [`examples/public_data/teapearce_csgo.md`](https://github.com/aintropy-ai/kuration-sdk/blob/main/examples/public_data/teapearce_csgo.md).

The upstream Hugging Face repo stores recordings inside multi-gigabyte archives (no official tiny sample). The smallest practical route is to download one expert archive, extract one HDF5 member, and run only that recording:

```bash
pip install -U huggingface_hub
hf download TeaPearce/CounterStrike_Deathmatch dataset_dm_expert_othermaps.tar \
  --repo-type dataset --local-dir data/teapearce
tar -xf data/teapearce/dataset_dm_expert_othermaps.tar <one-member>
python examples/public_data/teapearce_csgo.py <extracted-recording.hdf5> --out out/ --n 12
```

That is the recipe the Aintropy dev team uses when running against real gameplay. This notebook keeps the same operator chain and the same runnable structure, just against a synthetic recording so nothing needs to be downloaded first.